# Análisis de ventas · enero–junio 2026
**Pregunta de negocio:** ¿qué regiones, productos y vendedores explican los ingresos del semestre, y dónde hay oportunidades?

**Datos:** 120 ventas de una distribuidora de tecnología y oficina (`../datos`).

**Herramientas:** Python, pandas, matplotlib.

## 1. Cargar los datos

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATOS = Path('..') / 'datos'
ventas = pd.read_csv(DATOS / 'ventas.csv', parse_dates=['fecha'])
productos = pd.read_csv(DATOS / 'productos.csv')
vendedores = pd.read_csv(DATOS / 'vendedores.csv')
ventas.head()

,id,fecha,vendedor_id,producto_id,region,canal,cantidad,precio_unit
0,1,2026-01-01,2,102,Norte,Tienda,3,320
1,2,2026-01-02,4,104,Este,Online,1,85
2,3,2026-01-02,3,108,Sur,Tienda,6,95
3,4,2026-01-05,2,105,Norte,Online,5,260
4,5,2026-01-06,2,108,Norte,Tienda,8,95


## 2. Revisar calidad: tipos, vacíos y duplicados
Antes de analizar, siempre se valida.

In [2]:
print(ventas.dtypes, '\n')
print('Filas:', len(ventas))
print('Vacíos por columna:\n', ventas.isna().sum())
print('Duplicados:', ventas.duplicated().sum())

id                      int64
fecha          datetime64[us]
vendedor_id             int64
producto_id             int64
region                    str
canal                     str
cantidad                int64
precio_unit             int64
dtype: object 

Filas: 120
Vacíos por columna:
 id             0
fecha          0
vendedor_id    0
producto_id    0
region         0
canal          0
cantidad       0
precio_unit    0
dtype: int64
Duplicados: 0


## 3. Unir tablas y crear columnas de negocio
`merge` es el JOIN de pandas.

In [3]:
df = (ventas
      .merge(productos.rename(columns={'id': 'producto_id', 'nombre': 'producto'}), on='producto_id')
      .merge(vendedores.rename(columns={'id': 'vendedor_id', 'nombre': 'vendedor', 'region': 'region_vendedor'}), on='vendedor_id'))
df['ingreso'] = df['cantidad'] * df['precio_unit']
df['margen'] = df['cantidad'] * (df['precio_unit'] - df['costo'])
df['mes'] = df['fecha'].dt.to_period('M')
df[['fecha', 'region', 'producto', 'vendedor', 'cantidad', 'ingreso', 'margen']].head()

,fecha,region,producto,vendedor,cantidad,ingreso,margen
0,2026-01-01,Norte,Monitor 27,Luis Pérez,3,960,330
1,2026-01-02,Este,Teclado mecánico,Carlos Ruiz,1,85,35
2,2026-01-02,Sur,Audífonos BT,María Gómez,6,570,240
3,2026-01-05,Norte,Silla ergonómica,Luis Pérez,5,1300,550
4,2026-01-06,Norte,Audífonos BT,Luis Pérez,8,760,320


## 4. KPIs del semestre

In [4]:
kpis = {
    'Ingreso total': df['ingreso'].sum(),
    'Margen total': df['margen'].sum(),
    'Margen %': round(df['margen'].sum() / df['ingreso'].sum() * 100, 1),
    'Ventas': len(df),
    'Ticket promedio': round(df['ingreso'].mean(), 2),
}
pd.Series(kpis)

Ingreso total      77355.00
Margen total       27772.00
Margen %              35.90
Ventas               120.00
Ticket promedio      644.62
dtype: float64

## 5. Ingreso por región

In [5]:
COLORES = ['#11A64B', '#E3122F', '#1E5BFF', '#F2C200']
por_region = df.groupby('region')['ingreso'].sum().sort_values(ascending=False)
ax = por_region.plot(kind='bar', color=COLORES, title='Ingreso por región', rot=0)
ax.set_ylabel('USD'); plt.tight_layout(); plt.savefig('graficos/ingreso_region.png', dpi=120); plt.show()
(por_region / por_region.sum() * 100).round(1)

C:\Users\ANALISTA TI-TRACKER\AppData\Local\Temp\ipykernel_22388\1996215375.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_ylabel('USD'); plt.tight_layout(); plt.savefig('graficos/ingreso_region.png', dpi=120); plt.show()


region
Norte    42.1
Sur      35.1
Este     13.6
Oeste     9.2
Name: ingreso, dtype: float64

## 6. Evolución mensual

In [6]:
mensual = df.groupby('mes')['ingreso'].sum()
ax = mensual.plot(marker='o', color='#1E5BFF', title='Ingreso mensual')
ax.set_ylabel('USD'); plt.tight_layout(); plt.savefig('graficos/ingreso_mensual.png', dpi=120); plt.show()
mensual.pct_change().mul(100).round(1)

C:\Users\ANALISTA TI-TRACKER\AppData\Local\Temp\ipykernel_22388\2244804579.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_ylabel('USD'); plt.tight_layout(); plt.savefig('graficos/ingreso_mensual.png', dpi=120); plt.show()


mes
2026-01     NaN
2026-02   -25.8
2026-03    35.6
2026-04   -24.7
2026-05     5.9
2026-06   -37.8
Freq: M, Name: ingreso, dtype: float64

## 7. Productos: volumen vs. margen

In [7]:
prod = df.groupby('producto').agg(unidades=('cantidad', 'sum'), ingreso=('ingreso', 'sum'), margen=('margen', 'sum'))
prod['margen_pct'] = (prod['margen'] / prod['ingreso'] * 100).round(1)
prod = prod.sort_values('margen', ascending=False)
ax = prod['margen'].plot(kind='barh', color='#11A64B', title='Margen por producto')
ax.invert_yaxis(); plt.tight_layout(); plt.savefig('graficos/margen_producto.png', dpi=120); plt.show()
prod

C:\Users\ANALISTA TI-TRACKER\AppData\Local\Temp\ipykernel_22388\1538020289.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.invert_yaxis(); plt.tight_layout(); plt.savefig('graficos/margen_producto.png', dpi=120); plt.show()


,unidades,ingreso,margen,margen_pct
producto,,,,
Silla ergonómica,44,11440,4840,42.3
Impresora láser,52,10920,3640,33.3
Tablet 10,30,11400,3300,28.9
Teclado mecánico,82,6970,2870,41.2
Laptop Pro 14,9,10800,2700,25.0
Monitor 27,23,7360,2530,34.4
Webcam HD,90,5400,2520,46.7
Escritorio ajustable,13,6240,2340,37.5
Audífonos BT,55,5225,2200,42.1


## 8. Cumplimiento de metas por vendedor

In [8]:
vend = df.groupby(['vendedor', 'meta'])['ingreso'].sum().reset_index()
vend['cumplimiento_%'] = (vend['ingreso'] / vend['meta'] * 100).round(1)
vend = vend.sort_values('cumplimiento_%', ascending=False)
colores = ['#11A64B' if c >= 100 else '#E3122F' for c in vend['cumplimiento_%']]
ax = vend.plot(x='vendedor', y='cumplimiento_%', kind='bar', color=colores, legend=False, rot=20, title='Cumplimiento de meta (%)')
ax.axhline(100, color='#F2C200', linestyle='--'); plt.tight_layout(); plt.savefig('graficos/cumplimiento_meta.png', dpi=120); plt.show()
vend

C:\Users\ANALISTA TI-TRACKER\AppData\Local\Temp\ipykernel_22388\3842033388.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.axhline(100, color='#F2C200', linestyle='--'); plt.tight_layout(); plt.savefig('graficos/cumplimiento_meta.png', dpi=120); plt.show()


,vendedor,meta,ingreso,cumplimiento_%
3,Luis Pérez,18000,23920,132.9
2,Jorge Medina,17000,15775,92.8
4,María Gómez,20000,11400,57.0
1,Carlos Ruiz,20000,10510,52.6
0,Ana Torres,22000,8660,39.4
5,Sofía Díaz,21000,7090,33.8


## 9. Tabla dinámica: región × canal

In [9]:
pd.pivot_table(df, index='region', columns='canal', values='ingreso', aggfunc='sum', margins=True, margins_name='Total')

canal,Online,Tienda,Total
region,,,
Este,4735,5775,10510
Norte,14270,18310,32580
Oeste,1575,5515,7090
Sur,10405,16770,27175
Total,30985,46370,77355


## 10. Exportar el resumen a Excel

In [10]:
with pd.ExcelWriter('resumen_ventas.xlsx') as xls:
    pd.Series(kpis, name='valor').to_frame().to_excel(xls, sheet_name='KPIs')
    por_region.to_frame().to_excel(xls, sheet_name='Region')
    prod.to_excel(xls, sheet_name='Productos')
    vend.to_excel(xls, sheet_name='Vendedores', index=False)
print('resumen_ventas.xlsx creado')

resumen_ventas.xlsx creado


## Conclusiones
1. **Norte y Sur concentran el 77 % del ingreso.** Este y Oeste tienen espacio para crecer.
2. **Junio cayó 38 % frente a mayo**: hay que revisar stock y actividad comercial del cierre del semestre.
3. **Accesorios tiene el mejor margen % (44,5 %)** aunque menor ingreso: conviene impulsarlo en combos.
4. **Solo 1 de 6 vendedores superó su meta.** Las metas de Ana Torres y Sofía Díaz parecen desalineadas con su región.

**Recomendación:** priorizar campañas en Este/Oeste con productos de alto margen (Silla ergonómica, Webcam HD) y revisar las metas por región.